# Datavision EDA Summary

This notebook analyzes the prebuilt datavision dataset only (no raw table loading). It includes schema summary, missingness/zero rates, low variance checks, per-table prefix stats (using `FEATURE_TABLES`), label prevalence, and flagged feature candidates.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for path in candidates:
        dataset_path = path / "outputs" / "datavision_weekly_2025.parquet"
        if dataset_path.exists():
            return path
        if (path / "src").exists() and (path / "outputs").exists():
            return path
    raise FileNotFoundError("Could not locate repo root with outputs/datavision_weekly_2025.parquet")


root = find_repo_root(Path.cwd())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

dataset_path = root / "outputs" / "datavision_weekly_2025.parquet"
df = pd.read_parquet(dataset_path)

try:
    from src.data.build_dataset import FEATURE_TABLES
except Exception:
    FEATURE_TABLES = [
        "adl_responses",
        "care_plans",
        "diagnoses",
        "document_tags",
        "factors",
        "gg_responses",
        "hospital_admissions",
        "hospital_transfers",
        "incidents",
        "injuries",
        "lab_reports",
        "medications",
        "physician_orders",
        "vitals",
    ]

df.shape

(39770, 53)

## Schema Summary

In [2]:
schema = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "nulls": df.isna().sum().values,
    }
)
schema["null_rate"] = (schema["nulls"] / len(df)).round(4)
schema.sort_values(["null_rate", "column"], ascending=[False, True]).head(30)

,column,dtype,non_null,nulls,null_rate
6,adl_responses_days_since_last,float64,2252,37518,0.9434
22,gg_responses_days_since_last,float64,3597,36173,0.9096
42,medications_days_since_last,float64,8021,31749,0.7983
18,document_tags_days_since_last,float64,8679,31091,0.7818
30,hospital_transfers_days_since_last,float64,11711,28059,0.7055
38,lab_reports_days_since_last,float64,16009,23761,0.5975
26,hospital_admissions_days_since_last,float64,16991,22779,0.5728
34,incidents_days_since_last,float64,17838,21932,0.5515
50,vitals_days_since_last,float64,29407,10363,0.2606
46,physician_orders_days_since_last,float64,29487,10283,0.2586


## Missingness and Zero Rates

In [3]:
row_count = len(df)
missing_rate = df.isna().mean().rename("missing_rate")

numeric_cols = df.select_dtypes(include=["number"]).columns
zero_rate = pd.Series(dtype=float, name="zero_rate")
if len(numeric_cols) > 0:
    zero_rate = (df[numeric_cols] == 0).mean().rename("zero_rate")

missing_zero = (
    pd.concat([missing_rate, zero_rate], axis=1)
    .fillna(0)
    .sort_values(["missing_rate", "zero_rate"], ascending=False)
)
missing_zero.head(30)

,missing_rate,zero_rate
adl_responses_days_since_last,0.943374,0.004803
gg_responses_days_since_last,0.909555,0.008449
medications_days_since_last,0.798315,0.011642
document_tags_days_since_last,0.781770,0.008348
hospital_transfers_days_since_last,0.705532,0.000126
lab_reports_days_since_last,0.597460,0.000679
hospital_admissions_days_since_last,0.572768,0.000075
incidents_days_since_last,0.551471,0.000553
vitals_days_since_last,0.260573,0.062409
physician_orders_days_since_last,0.258562,0.003093


## Near-Constant / Low Variance

In [4]:
low_variance_rows = []

for col in df.columns:
    series = df[col]
    nunique = series.nunique(dropna=True)
    if pd.api.types.is_numeric_dtype(series):
        variance = series.var(skipna=True)
        if nunique <= 1 or (variance is not None and variance <= 1e-12):
            low_variance_rows.append(
                {
                    "column": col,
                    "type": "numeric",
                    "nunique": int(nunique),
                    "variance": float(variance) if pd.notna(variance) else None,
                }
            )
    else:
        if nunique <= 1:
            low_variance_rows.append(
                {
                    "column": col,
                    "type": "non_numeric",
                    "nunique": int(nunique),
                    "variance": None,
                }
            )

if low_variance_rows:
    low_variance = pd.DataFrame(low_variance_rows).sort_values(
        ["nunique", "column"], ascending=[True, True]
    )
else:
    low_variance = pd.DataFrame(
        columns=["column", "type", "nunique", "variance"]
    )

low_variance.head(30)

KeyError: 'nunique'

## Per-Table Prefix Stats (FEATURE_TABLES)

In [ ]:
prefix_rows = []

for prefix in FEATURE_TABLES:
    prefix_key = f"{prefix}_"
    cols = [c for c in df.columns if c.startswith(prefix_key)]
    if not cols:
        prefix_rows.append(
            {
                "table": prefix,
                "columns": 0,
                "missing_rate_mean": None,
                "zero_rate_mean": None,
            }
        )
        continue

    missing_mean = df[cols].isna().mean().mean()
    
    numeric_cols = df[cols].select_dtypes(include=["number"]).columns
    if len(numeric_cols) > 0:
        zero_mean = (df[numeric_cols] == 0).mean().mean()
    else:
        zero_mean = None

    prefix_rows.append(
        {
            "table": prefix,
            "columns": len(cols),
            "missing_rate_mean": float(missing_mean),
            "zero_rate_mean": float(zero_mean) if zero_mean is not None else None,
        }
    )

prefix_stats = pd.DataFrame(prefix_rows).sort_values(["columns", "table"], ascending=[False, True])
prefix_stats

## Label Prevalence

In [ ]:
label_cols = [c for c in ["label_fall_30d", "label_rth_30d"] if c in df.columns]

label_stats = []
for col in label_cols:
    series = df[col]
    label_stats.append(
        {
            "label": col,
            "prevalence": float(series.mean()) if len(series) > 0 else None,
            "positives": int(series.sum()) if len(series) > 0 else None,
            "rows": int(len(series)),
        }
    )

pd.DataFrame(label_stats)

## Flagged Candidates

In [ ]:
flags = []

missing_rate = df.isna().mean()
numeric_cols = df.select_dtypes(include=["number"]).columns
zero_rate = pd.Series(0.0, index=df.columns)
zero_rate.loc[numeric_cols] = (df[numeric_cols] == 0).mean()

for col in df.columns:
    series = df[col]
    nunique = series.nunique(dropna=True)
    variance = series.var(skipna=True) if pd.api.types.is_numeric_dtype(series) else None

    reasons = []
    if missing_rate[col] >= 0.9:
        reasons.append("missing>=0.90")
    if col in numeric_cols and zero_rate[col] >= 0.95:
        reasons.append("zero>=0.95")
    if nunique <= 1:
        reasons.append("nunique<=1")
    if variance is not None and variance <= 1e-12:
        reasons.append("variance<=1e-12")

    if reasons:
        flags.append(
            {
                "column": col,
                "missing_rate": float(missing_rate[col]),
                "zero_rate": float(zero_rate[col]) if col in numeric_cols else None,
                "nunique": int(nunique),
                "variance": float(variance) if variance is not None else None,
                "reasons": ";".join(reasons),
            }
        )

flagged = pd.DataFrame(flags).sort_values(
    ["missing_rate", "zero_rate", "nunique", "column"], ascending=[False, False, True, True]
)
flagged.head(50)